# DeFi Strategy Benchmark: Historical Backtest (Jan 2023 – Jun 2026)

## Objective
Benchmark 5 DeFi trading strategies on AAVE V3 historical data to evaluate relative performance, risk-adjusted returns, and robustness across different market regimes.

## Strategies

| # | Strategy | Core Mechanism | Key Risk |
|---|----------|----------------|----------|
| 1 | **Current Carry Trade** | USDC supply + WETH borrow (LTV 70%) | ETH price spike → liquidation |
| 2 | **Taleb Barbell** | 90% safe supply + 10% carry (LTV 70%) | Limited upside but protected capital |
| 3 | **Pure Supply & Demand** | On-chain utilization signal, LTV 50% | Signal lag, missed spreads |
| 4 | **Black-Litterman Portfolio** | Bayesian TVL-weighted carry, weekly rebalance | Model error in views |
| 5 | **Momentum & Factor Rotation** | Multi-factor score, weekly top-2 selection | Factor crowding, whipsaws |

**Capital**: $100,000 USDC  
**Period**: Jan 2023 – Jun 2026 (~1,222 trading days)  
**Data**: AAVE V3 Ethereum hourly → daily aggregated

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports loaded successfully')
print(f'Numpy {np.__version__}, Pandas {pd.__version__}')

Imports loaded successfully
Numpy 2.4.4, Pandas 2.3.3


In [2]:
# ─── Constants & Configuration ───────────────────────────────────────────────
INITIAL_CAPITAL  = 100_000      # USD
LTV_CARRY        = 0.70         # Strategy 1 LTV
LTV_BARBELL_AGG  = 0.70         # Strategy 2 aggressive leg LTV
LTV_SD           = 0.50         # Strategy 3 LTV (more conservative)
LTV_BL           = 0.60         # Strategy 4 LTV
LTV_MOM          = 0.40         # Strategy 5 LTV
LIQ_THRESHOLD    = 0.825        # AAVE V3 USDC liquidation threshold
LIQ_PENALTY      = 0.05         # 5% liquidation penalty
RF_ANNUAL        = 0.05         # 5% risk-free rate
BARBELL_SAFE_FRAC = 0.90        # 90% in safe supply
BARBELL_AGG_FRAC  = 0.10        # 10% in aggressive carry

# Output directory
OUT_DIR = Path('.')             # save plots in current directory

# Plot style
plt.style.use('dark_background')
COLORS = ['#00d4ff', '#ff6b35', '#7fff00', '#ff69b4', '#ffd700']
STRATEGY_NAMES = [
    'Carry Trade',
    'Taleb Barbell',
    'Supply & Demand',
    'Black-Litterman',
    'Momentum Rotation',
]

print('Constants configured')

Constants configured


In [3]:
# ─── Data Loading ──────────────────────────────────────────────────────────
DATA_DIR = Path('../data/OLD_AAVE/')
RAW_DIR  = Path('../data/raw/')

def load_aave(asset):
    fpath = DATA_DIR / f'aave_v3_{asset}_eth.parquet'
    if not fpath.exists():
        raise FileNotFoundError(f'Missing: {fpath}')
    return pd.read_parquet(fpath)

def daily_resample(df):
    return df.resample('D').agg({
        'lender_variable_apr':   'mean',
        'borrower_variable_apr': 'mean',
        'TVL_USD':               'last',
        'supplied_USD':          'last',
        'borrowed_USD':          'last',
        'liquidations':          'sum',
    }).dropna(subset=['lender_variable_apr'])

def load_coin_price(fname):
    fpath = RAW_DIR / fname
    if not fpath.exists():
        raise FileNotFoundError(f'Missing: {fpath}')
    df = pd.read_csv(fpath, parse_dates=['snapped_at'])
    df['date'] = pd.to_datetime(df['snapped_at']).dt.tz_localize(None).dt.normalize()
    return df.set_index('date')['price'].sort_index()

try:
    d_usdc = daily_resample(load_aave('usdc'))
    d_weth = daily_resample(load_aave('weth'))
    d_dai  = daily_resample(load_aave('dai'))
    d_usdt = daily_resample(load_aave('usdt'))
    eth_usd  = load_coin_price('eth-usd-max.csv')
    usdc_usd = load_coin_price('usdc-usd-max.csv')
    weth_usdc = eth_usd / usdc_usd
    print('All data files loaded successfully')
except FileNotFoundError as e:
    print(f'ERROR: {e}')
    raise

# ── Build daily DataFrame ──
daily = pd.DataFrame({
    'usdc_supply':   d_usdc['lender_variable_apr']   / 100,
    'weth_borrow':   d_weth['borrower_variable_apr'] / 100,
    'dai_supply':    d_dai['lender_variable_apr']    / 100,
    'usdt_supply':   d_usdt['lender_variable_apr']   / 100,
    'usdc_tvl':      d_usdc['TVL_USD'],
    'weth_tvl':      d_weth['TVL_USD'],
    'dai_tvl':       d_dai['TVL_USD'],
    'usdc_supplied': d_usdc['supplied_USD'],
    'usdc_borrowed': d_usdc['borrowed_USD'],
    'eth_price':     weth_usdc,
}).dropna(subset=['usdc_supply', 'weth_borrow', 'eth_price'])

daily['eth_return']   = daily['eth_price'].pct_change()
daily['carry_spread'] = daily['usdc_supply'] - daily['weth_borrow']
daily['usdc_util']    = daily['usdc_borrowed'] / daily['usdc_supplied'].replace(0, np.nan)
daily = daily.dropna(subset=['eth_return'])

print(f'Daily data: {daily.shape[0]} rows | {daily.index[0].date()} to {daily.index[-1].date()}')
print(f'Mean USDC supply APR : {daily["usdc_supply"].mean()*100:.2f}%')
print(f'Mean WETH borrow APR : {daily["weth_borrow"].mean()*100:.2f}%')
print(f'Mean carry spread    : {daily["carry_spread"].mean()*100:.2f}%  ({(daily["carry_spread"]>0).mean()*100:.1f}% days positive)')

All data files loaded successfully
Daily data: 1222 rows | 2023-01-28 to 2026-06-02
Mean USDC supply APR : 4.80%
Mean WETH borrow APR : 2.85%
Mean carry spread    : 1.95%  (74.8% days positive)


In [4]:
# ─── Feature Engineering (30d rolling + regime features) ─────────────────────
W30 = 30
W7  = 7
W14 = 14

# Utilization signal for Strategy 3
daily['util_30d_mean'] = daily['usdc_util'].rolling(W30, min_periods=W7).mean()
daily['util_30d_std']  = daily['usdc_util'].rolling(W30, min_periods=W7).std()

# ETH volatility
daily['eth_vol_30d']   = daily['eth_return'].rolling(W30, min_periods=W7).std() * np.sqrt(365)

# Carry spread rolling stats
daily['cs_30d_mean']   = daily['carry_spread'].rolling(W30, min_periods=W7).mean()
daily['cs_30d_std']    = daily['carry_spread'].rolling(W30, min_periods=W7).std()

# Factors for Strategy 5
daily['yield_mom']     = (daily['usdc_supply'] - daily['usdc_supply'].rolling(W30, min_periods=W7).mean()) / \
                          (daily['usdc_supply'].rolling(W30, min_periods=W7).std() + 1e-6)
daily['tvl_mom']       = daily['usdc_tvl'].pct_change(W14)
daily['borrow_mom']    = daily['usdc_borrowed'].pct_change(W7)
daily['vol_regime']    = -(daily['eth_return'].rolling(W30, min_periods=W7).std() * np.sqrt(365))
daily['carry_mom']     = daily['carry_spread'] - daily['carry_spread'].rolling(W30, min_periods=W7).mean()

# DAI/USDT factor signals (reuse same momentum structure)
daily['dai_yield_mom'] = (daily['dai_supply'] - daily['dai_supply'].rolling(W30, min_periods=W7).mean()) / \
                          (daily['dai_supply'].rolling(W30, min_periods=W7).std() + 1e-6)
daily['usdt_yield_mom']= (daily['usdt_supply'] - daily['usdt_supply'].rolling(W30, min_periods=W7).mean()) / \
                          (daily['usdt_supply'].rolling(W30, min_periods=W7).std() + 1e-6)

# BL: TVL-proportional market weights (weekly resample aligned)
daily['total_tvl']     = daily['usdc_tvl'] + daily['dai_tvl'].fillna(0) + daily['weth_tvl']

print('Feature engineering complete')
print(f'Rolling features computed | NaN rows from warmup: {daily[["yield_mom","tvl_mom"]].isna().any(axis=1).sum()}')

Feature engineering complete
Rolling features computed | NaN rows from warmup: 14


In [5]:
# ─── Regime Masks ────────────────────────────────────────────────────────────
idx = daily.index

def date_mask(start, end):
    # Return a pandas Series (boolean) so .shift() works for regime shading
    arr = (idx >= start) & (idx <= end)
    return pd.Series(arr, index=idx)

regimes = {
    'bull':     date_mask('2023-10-01', '2023-12-31') |
                date_mask('2024-01-01', '2024-03-31') |
                date_mask('2024-10-01', '2024-12-31'),
    'bear':     date_mask('2023-01-27', '2023-06-15') |
                date_mask('2024-06-01', '2024-08-31'),
    'sideways': date_mask('2023-07-01', '2023-09-30') |
                date_mask('2024-04-01', '2024-06-30'),
    'crisis':   date_mask('2023-03-01', '2023-03-31') |
                date_mask('2024-08-01', '2024-08-15'),
    'high_vol': date_mask('2023-03-01', '2023-04-15') |
                date_mask('2024-08-01', '2024-09-15'),
}

# Regime colors for plotting
REGIME_COLORS = {
    'bull':     ('#00ff41', 0.12),
    'bear':     ('#ff4444', 0.12),
    'sideways': ('#ffaa00', 0.10),
    'crisis':   ('#ff00ff', 0.20),
    'high_vol': ('#00aaff', 0.10),
}

for name, mask in regimes.items():
    print(f'  {name:10s}: {mask.sum():4d} days')

  bull      :  275 days
  bear      :  231 days
  sideways  :  183 days
  crisis    :   46 days
  high_vol  :   92 days


In [6]:
# ─── Performance Metrics Helper ──────────────────────────────────────────────
def compute_metrics(returns, rf_annual=RF_ANNUAL, capital=INITIAL_CAPITAL, name='Strategy'):
    """
    Compute 14 performance metrics from a Series of daily fractional returns.
    Returns a dict.
    """
    r = returns.dropna()
    if len(r) < 5:
        return {m: np.nan for m in [
            'Cumulative Return', 'Annualized Return', 'Annualized Volatility',
            'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Calmar Ratio',
            'Win Rate', 'Profit Factor', 'VaR 95%', 'CVaR 95%',
            'Skewness', 'Kurtosis', 'Omega Ratio']}

    n = len(r)
    years = n / 365
    rf_daily = rf_annual / 365

    # Equity curve
    eq = (1 + r).cumprod()

    # 1. Cumulative Return
    cum_ret = eq.iloc[-1] - 1

    # 2. Annualized Return (CAGR)
    ann_ret = (1 + cum_ret) ** (1 / years) - 1

    # 3. Annualized Volatility
    ann_vol = r.std() * np.sqrt(365)

    # 4. Sharpe Ratio
    excess = r - rf_daily
    sharpe = (excess.mean() / (r.std() + 1e-10)) * np.sqrt(365)

    # 5. Sortino Ratio
    downside = r[r < rf_daily]
    dd_std = np.sqrt((downside**2).mean()) * np.sqrt(365) if len(downside) > 0 else 1e-10
    sortino = (ann_ret - rf_annual) / (dd_std + 1e-10)

    # 6. Max Drawdown
    roll_max = eq.cummax()
    drawdown = (eq - roll_max) / roll_max
    max_dd = drawdown.min()

    # 7. Calmar Ratio
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-10 else np.nan

    # 8. Win Rate
    win_rate = (r > 0).mean()

    # 9. Profit Factor
    gains  = r[r > 0].sum()
    losses = abs(r[r < 0].sum())
    profit_factor = gains / losses if losses > 1e-10 else np.nan

    # 10. VaR 95% (as USD loss, positive = loss)
    var_95 = -np.percentile(r, 5) * capital

    # 11. CVaR 95%
    cvar_95 = -r[r <= np.percentile(r, 5)].mean() * capital

    # 12. Skewness
    skew = stats.skew(r)

    # 13. Kurtosis (excess)
    kurt = stats.kurtosis(r)

    # 14. Omega Ratio
    threshold = rf_daily
    above = (r[r > threshold] - threshold).sum()
    below = (threshold - r[r < threshold]).sum()
    omega = above / below if below > 1e-10 else np.nan

    return {
        'Cumulative Return':    cum_ret,
        'Annualized Return':    ann_ret,
        'Annualized Volatility': ann_vol,
        'Sharpe Ratio':         sharpe,
        'Sortino Ratio':        sortino,
        'Max Drawdown':         max_dd,
        'Calmar Ratio':         calmar,
        'Win Rate':             win_rate,
        'Profit Factor':        profit_factor,
        'VaR 95% ($)':          var_95,
        'CVaR 95% ($)':         cvar_95,
        'Skewness':             skew,
        'Kurtosis':             kurt,
        'Omega Ratio':          omega,
    }


def equity_curve(returns):
    """Equity curve normalized to 1.0 start."""
    return (1 + returns.fillna(0)).cumprod()


def drawdown_series(returns):
    eq = equity_curve(returns)
    roll_max = eq.cummax()
    return (eq - roll_max) / roll_max


print('Metrics helper functions defined')

Metrics helper functions defined


## Strategy 1: Current Carry Trade
**Mechanism**: Deposit USDC as collateral, borrow WETH at LTV 70%, receive supply interest minus borrow cost. Enter only when carry spread > 0.

In [7]:
# ─── Strategy 1: Current Carry Trade ─────────────────────────────────────────
def strategy_carry_trade(df, initial_capital=INITIAL_CAPITAL, ltv=LTV_CARRY):
    """
    USDC supply + WETH borrow at LTV 70%.
    C = collateral (USDC), W = WETH tokens borrowed
    HF = C * 0.825 / (W * eth_price)  → liquidated if HF < 1
    Entry: carry_spread > 0 AND not in cooldown after liquidation.
    Post-liquidation cooldown: wait for spread to turn <=0 then back >0 before re-entering.
    Returns daily fractional returns Series.
    """
    dates    = df.index
    usdc_s   = df['usdc_supply'].values
    weth_b   = df['weth_borrow'].values
    eth_p    = df['eth_price'].values
    carry_sp = df['carry_spread'].values

    rets      = np.zeros(len(df))
    C         = float(initial_capital)
    W         = 0.0
    in_carry  = False
    cooldown  = False   # post-liquidation: wait for spread reset before re-entering

    for i in range(1, len(df)):
        if C <= 0:
            rets[i] = 0.0
            continue

        prev_equity = C - W * eth_p[i-1]

        # Manage cooldown: cleared when spread turns non-positive
        if cooldown and carry_sp[i] <= 0:
            cooldown = False

        # Entry / exit
        if carry_sp[i] > 0 and not in_carry and not cooldown:
            W = ltv * C / eth_p[i-1]
            in_carry = True
        elif carry_sp[i] <= 0 and in_carry:
            W = 0.0
            in_carry = False

        # Accrue interest
        C_new = C * (1 + usdc_s[i] / 365)
        W_new = W * (1 + weth_b[i] / 365)

        # Liquidation check
        D_new = W_new * eth_p[i]
        if W_new > 0:
            hf = C_new * LIQ_THRESHOLD / (D_new + 1e-10)
            if hf < 1.0:
                remaining = max(0.0, C_new - D_new * (1 + LIQ_PENALTY))
                ret = remaining / max(prev_equity, 1e-6) - 1
                C = remaining
                W = 0.0
                in_carry = False
                cooldown = True   # enter cooldown: wait for spread reset
                rets[i] = ret
                continue

        # Normal return
        if in_carry:
            new_equity = C_new - W_new * eth_p[i]
            ret = new_equity / max(prev_equity, 1e-6) - 1
        else:
            # Pure supply return
            ret = usdc_s[i] / 365
            new_equity = C_new

        C = C_new
        W = W_new
        rets[i] = ret

    return pd.Series(rets[1:], index=dates[1:], name='Carry Trade')


s1_returns = strategy_carry_trade(daily)
s1_metrics = compute_metrics(s1_returns, name='Carry Trade')
print('Strategy 1 — Carry Trade')
print(f"  Cumulative Return : {s1_metrics['Cumulative Return']*100:.2f}%")
print(f"  Annualized Return : {s1_metrics['Annualized Return']*100:.2f}%")
print(f"  Sharpe Ratio      : {s1_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown      : {s1_metrics['Max Drawdown']*100:.2f}%")
print(f"  Final equity      : ${INITIAL_CAPITAL * (1+s1_metrics['Cumulative Return']):,.0f}")

Strategy 1 — Carry Trade
  Cumulative Return : -100.00%
  Annualized Return : -100.00%
  Sharpe Ratio      : -3.310
  Max Drawdown      : -100.00%
  Final equity      : $0


## Strategy 2: Taleb Barbell
**Mechanism**: 90% in pure USDC supply (safe), 10% in carry trade with LTV 70% (aggressive). Liquidation affects aggressive leg only.

In [8]:
# ─── Strategy 2: Taleb Barbell ────────────────────────────────────────────────
def strategy_barbell(df, initial_capital=INITIAL_CAPITAL,
                     safe_frac=BARBELL_SAFE_FRAC, ltv=LTV_BARBELL_AGG):
    """
    90% pure USDC supply  (E_safe)
    10% aggressive carry  (E_agg_collateral, W_agg tokens)
    equity = E_safe + (E_agg_collateral - W_agg * eth_price)
    daily return = delta_equity / initial_total_equity
    """
    dates    = df.index
    usdc_s   = df['usdc_supply'].values
    weth_b   = df['weth_borrow'].values
    eth_p    = df['eth_price'].values
    carry_sp = df['carry_spread'].values

    rets = np.zeros(len(df))

    E_safe  = float(initial_capital) * safe_frac
    C_agg   = float(initial_capital) * (1 - safe_frac)   # aggressive collateral
    W_agg   = C_agg * ltv / eth_p[0]                      # WETH borrowed on agg leg
    total_init = initial_capital

    for i in range(1, len(df)):
        prev_total = E_safe + max(0.0, C_agg - W_agg * eth_p[i-1])

        # Accrue safe supply
        E_safe_new = E_safe * (1 + usdc_s[i] / 365)

        # Accrue aggressive leg
        C_agg_new = C_agg * (1 + usdc_s[i] / 365)
        W_agg_new = W_agg * (1 + weth_b[i] / 365)

        # Liquidation check on aggressive leg
        D_agg = W_agg_new * eth_p[i]
        if W_agg_new > 0:
            hf_agg = C_agg_new * LIQ_THRESHOLD / (D_agg + 1e-10)
            if hf_agg < 1.0:
                agg_remaining = max(0.0, C_agg_new - D_agg * (1 + LIQ_PENALTY))
                C_agg_new = agg_remaining
                W_agg_new = 0.0
                if carry_sp[i] > 0 and C_agg_new > 0:
                    W_agg_new = C_agg_new * ltv / eth_p[i]

        new_total = E_safe_new + max(0.0, C_agg_new - W_agg_new * eth_p[i])
        rets[i] = (new_total - prev_total) / max(prev_total, 1e-6)

        E_safe = E_safe_new
        C_agg  = C_agg_new
        W_agg  = W_agg_new

    return pd.Series(rets[1:], index=dates[1:], name='Taleb Barbell')


s2_returns = strategy_barbell(daily)
s2_metrics = compute_metrics(s2_returns, name='Taleb Barbell')
print('Strategy 2 — Taleb Barbell')
print(f"  Cumulative Return : {s2_metrics['Cumulative Return']*100:.2f}%")
print(f"  Annualized Return : {s2_metrics['Annualized Return']*100:.2f}%")
print(f"  Sharpe Ratio      : {s2_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown      : {s2_metrics['Max Drawdown']*100:.2f}%")
print(f"  Final equity      : ${INITIAL_CAPITAL * (1+s2_metrics['Cumulative Return']):,.0f}")

Strategy 2 — Taleb Barbell
  Cumulative Return : 15.10%
  Annualized Return : 4.29%
  Sharpe Ratio      : -0.709
  Max Drawdown      : -2.58%
  Final equity      : $115,102


## Strategy 3: Pure Supply & Demand
**Mechanism**: Enter carry trade (LTV 50%) only when utilization exceeds its 30-day rolling mean by 0.5 standard deviations. Otherwise, hold pure USDC supply.

In [9]:
# ─── Strategy 3: Pure Supply & Demand ────────────────────────────────────────
def strategy_supply_demand(df, initial_capital=INITIAL_CAPITAL, ltv=LTV_SD):
    """
    Utilization signal: util > util_30d_mean + 0.5*util_30d_std → carry (LTV 50%)
    No signal: pure USDC supply.
    Post-liquidation cooldown: wait for signal to drop before re-entering.
    """
    dates   = df.index
    usdc_s  = df['usdc_supply'].values
    weth_b  = df['weth_borrow'].values
    eth_p   = df['eth_price'].values
    util    = df['usdc_util'].ffill().values
    u_mean  = df['util_30d_mean'].ffill().values
    u_std   = df['util_30d_std'].fillna(0.01).values

    rets     = np.zeros(len(df))
    C        = float(initial_capital)
    W        = 0.0
    in_carry = False
    cooldown = False

    for i in range(1, len(df)):
        if C <= 0:
            rets[i] = 0.0
            continue

        # Signal
        if not np.isnan(u_mean[i]) and u_std[i] > 0:
            signal = util[i] > (u_mean[i] + 0.5 * u_std[i])
        else:
            signal = False

        # Cooldown: clear when signal drops
        if cooldown and not signal:
            cooldown = False

        # Entry / exit carry
        if signal and not in_carry and not cooldown:
            W = ltv * C / eth_p[i-1]
            in_carry = True
        elif not signal and in_carry:
            W = 0.0
            in_carry = False

        # Accrue
        C_new = C * (1 + usdc_s[i] / 365)
        W_new = W * (1 + weth_b[i] / 365)

        # Liquidation
        if W_new > 0:
            D_new = W_new * eth_p[i]
            hf = C_new * LIQ_THRESHOLD / (D_new + 1e-10)
            if hf < 1.0:
                remaining = max(0.0, C_new - D_new * (1 + LIQ_PENALTY))
                prev_eq = max(C - W * eth_p[i-1], 1e-6)
                rets[i] = remaining / prev_eq - 1
                C = remaining
                W = 0.0
                in_carry = False
                cooldown = True
                continue

        prev_eq = C - W * eth_p[i-1] if in_carry else C
        new_eq  = C_new - W_new * eth_p[i] if in_carry else C_new
        rets[i] = new_eq / max(prev_eq, 1e-6) - 1

        C = C_new
        W = W_new

    return pd.Series(rets[1:], index=dates[1:], name='Supply & Demand')


s3_returns = strategy_supply_demand(daily)
s3_metrics = compute_metrics(s3_returns, name='Supply & Demand')
print('Strategy 3 — Pure Supply & Demand')
print(f"  Cumulative Return : {s3_metrics['Cumulative Return']*100:.2f}%")
print(f"  Annualized Return : {s3_metrics['Annualized Return']*100:.2f}%")
print(f"  Sharpe Ratio      : {s3_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown      : {s3_metrics['Max Drawdown']*100:.2f}%")
print(f"  Final equity      : ${INITIAL_CAPITAL * (1+s3_metrics['Cumulative Return']):,.0f}")

Strategy 3 — Pure Supply & Demand
  Cumulative Return : -73.81%
  Annualized Return : -33.01%
  Sharpe Ratio      : -0.773
  Max Drawdown      : -79.64%
  Final equity      : $26,185


## Strategy 4: Black-Litterman Portfolio
**Mechanism**: Bayesian combination of market-equilibrium TVL-proportional weights with an active "carry outperforms" view (60% confidence). Rebalance weekly.

In [10]:
# ─── Strategy 4: Black-Litterman Portfolio ────────────────────────────────────
def strategy_black_litterman(df, initial_capital=INITIAL_CAPITAL, ltv=LTV_BL):
    """
    3 assets:
      - USDC supply (r_usdc)
      - DAI supply  (r_dai)
      - WETH carry  (r_carry = usdc_supply/365 - ltv*(weth_borrow/365 + eth_daily_return))
    BL update with one active view: 'carry > pure supply by cs_30d_mean'
    Posterior weights proportional to BL expected returns, max 50% single asset.
    Rebalance every 7 days.
    """
    dates    = df.index
    usdc_s   = df['usdc_supply'].values
    dai_s    = df['dai_supply'].fillna(df['usdc_supply']).values
    weth_b   = df['weth_borrow'].values
    eth_ret  = df['eth_return'].values
    cs_30m   = df['cs_30d_mean'].fillna(df['carry_spread']).values
    usdc_tvl = df['usdc_tvl'].fillna(1e9).values
    dai_tvl  = df['dai_tvl'].fillna(1e7).values
    weth_tvl = df['weth_tvl'].fillna(1e8).values

    tau = 0.05    # BL tau parameter

    rets   = np.zeros(len(df))
    equity = float(initial_capital)

    # Initial TVL-proportional weights
    total_tvl = usdc_tvl[0] + dai_tvl[0] + weth_tvl[0]
    w = np.array([
        usdc_tvl[0] / total_tvl,
        dai_tvl[0]  / total_tvl,
        weth_tvl[0] / total_tvl,
    ])

    rebal_counter = 0

    for i in range(1, len(df)):
        if equity <= 0:
            rets[i] = 0.0
            continue

        # Daily asset returns
        r_usdc  = usdc_s[i] / 365
        r_dai   = dai_s[i]  / 365
        r_carry = r_usdc - ltv * (weth_b[i] / 365 + max(eth_ret[i], -0.30))

        asset_rets = np.array([r_usdc, r_dai, r_carry])

        # Portfolio return
        port_ret = np.dot(w, asset_rets)
        rets[i]  = port_ret
        equity  *= (1 + port_ret)

        # Weekly rebalance
        rebal_counter += 1
        if rebal_counter >= 7:
            rebal_counter = 0

            # Equilibrium weights from TVL
            t = usdc_tvl[i] + dai_tvl[i] + weth_tvl[i]
            Pi = np.array([
                usdc_tvl[i] / t * r_usdc * 365,
                dai_tvl[i]  / t * r_dai  * 365,
                weth_tvl[i] / t * r_carry * 365,
            ])

            # Rolling covariance (approximate from spread volatility)
            sigma2 = max(df['carry_spread'].iloc[max(0,i-30):i].var(), 1e-8)

            # BL view: carry outperforms pure supply by cs_30d_mean
            view_confidence = 0.60
            P   = np.array([0.0, 0.0, 1.0])   # carry asset
            q   = float(cs_30m[i]) if cs_30m[i] > 0 else 0.0  # view return
            omega = (1 - view_confidence) / view_confidence * tau * sigma2

            # Simplified BL posterior expected returns
            bl_adj = tau * sigma2 / (tau * sigma2 + omega) * (q - Pi[2])
            bl_ret = Pi.copy()
            bl_ret[2] += bl_adj

            # Weights proportional to expected returns, floor at 0
            bl_w = np.maximum(bl_ret, 0)
            if bl_w.sum() < 1e-10:
                bl_w = np.array([1/3, 1/3, 1/3])
            else:
                bl_w = bl_w / bl_w.sum()

            # Max 50% single asset
            bl_w = np.minimum(bl_w, 0.50)
            if bl_w.sum() < 1e-10:
                bl_w = np.array([1/3, 1/3, 1/3])
            else:
                bl_w = bl_w / bl_w.sum()

            w = bl_w

    return pd.Series(rets[1:], index=dates[1:], name='Black-Litterman')


s4_returns = strategy_black_litterman(daily)
s4_metrics = compute_metrics(s4_returns, name='Black-Litterman')
print('Strategy 4 — Black-Litterman Portfolio')
print(f"  Cumulative Return : {s4_metrics['Cumulative Return']*100:.2f}%")
print(f"  Annualized Return : {s4_metrics['Annualized Return']*100:.2f}%")
print(f"  Sharpe Ratio      : {s4_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown      : {s4_metrics['Max Drawdown']*100:.2f}%")
print(f"  Final equity      : ${INITIAL_CAPITAL * (1+s4_metrics['Cumulative Return']):,.0f}")

Strategy 4 — Black-Litterman Portfolio
  Cumulative Return : -4.08%
  Annualized Return : -1.24%
  Sharpe Ratio      : -0.117
  Max Drawdown      : -27.01%
  Final equity      : $95,924


## Strategy 5: Momentum & Factor Rotation
**Mechanism**: Score 4 assets (USDC supply, DAI supply, USDT supply, WETH carry) on 5 factors weekly. Select top-2 by score with equal weight. Reduce 50% during high ETH volatility.

In [11]:
# ─── Strategy 5: Momentum & Factor Rotation ───────────────────────────────────
def strategy_momentum_rotation(df, initial_capital=INITIAL_CAPITAL, ltv=LTV_MOM):
    """
    4 assets: USDC supply, DAI supply, USDT supply, WETH carry (LTV 40%).
    5 factors computed daily; top-2 selected weekly.
    Reduce weights 50% when eth_vol_30d > 0.50.
    """
    dates      = df.index
    usdc_s     = df['usdc_supply'].values
    dai_s      = df['dai_supply'].fillna(df['usdc_supply']).values
    usdt_s     = df['usdt_supply'].fillna(df['usdc_supply']).values
    weth_b     = df['weth_borrow'].values
    eth_ret    = df['eth_return'].values
    eth_vol    = df['eth_vol_30d'].fillna(0.50).values

    yield_mom  = df['yield_mom'].fillna(0.0).values
    tvl_mom    = df['tvl_mom'].fillna(0.0).values
    borrow_mom = df['borrow_mom'].fillna(0.0).values
    vol_regime = df['vol_regime'].fillna(-0.5).values
    carry_mom  = df['carry_mom'].fillna(0.0).values
    dai_ym     = df['dai_yield_mom'].fillna(0.0).values
    usdt_ym    = df['usdt_yield_mom'].fillna(0.0).values

    rets   = np.zeros(len(df))
    equity = float(initial_capital)

    # Current weights: equal start
    w_usdc, w_dai, w_usdt, w_carry = 0.25, 0.25, 0.25, 0.25
    rebal_counter = 0

    for i in range(1, len(df)):
        if equity <= 0:
            rets[i] = 0.0
            continue

        # Asset daily returns
        r_usdc  = usdc_s[i]  / 365
        r_dai   = dai_s[i]   / 365
        r_usdt  = usdt_s[i]  / 365
        r_carry = r_usdc - ltv * (weth_b[i] / 365 + max(eth_ret[i], -0.30))

        # Portfolio daily return
        port_ret = (w_usdc * r_usdc + w_dai * r_dai +
                    w_usdt * r_usdt + w_carry * r_carry)
        rets[i]  = port_ret
        equity  *= (1 + port_ret)

        # Weekly rebalance
        rebal_counter += 1
        if rebal_counter >= 7:
            rebal_counter = 0

            # Score each asset
            sc_usdc  = (0.4 * yield_mom[i] + 0.3 * tvl_mom[i] +
                        0.2 * borrow_mom[i] - 0.1 * vol_regime[i])
            sc_dai   = (0.4 * dai_ym[i]    + 0.3 * tvl_mom[i] +
                        0.2 * borrow_mom[i] - 0.1 * vol_regime[i])
            sc_usdt  = (0.4 * usdt_ym[i]   + 0.3 * tvl_mom[i] +
                        0.2 * borrow_mom[i] - 0.1 * vol_regime[i])
            sc_carry = (0.3 * carry_mom[i] + 0.3 * vol_regime[i] +
                        0.2 * borrow_mom[i] + 0.2 * yield_mom[i])

            scores = np.array([sc_usdc, sc_dai, sc_usdt, sc_carry])
            top2   = np.argsort(scores)[-2:]

            # Equal weight top-2
            new_w  = np.zeros(4)
            new_w[top2] = 0.50

            # Reduce 50% in high vol regime
            if eth_vol[i] > 0.50:
                new_w *= 0.50   # rest in cash (0% return proxy)

            w_usdc, w_dai, w_usdt, w_carry = new_w

    return pd.Series(rets[1:], index=dates[1:], name='Momentum Rotation')


s5_returns = strategy_momentum_rotation(daily)
s5_metrics = compute_metrics(s5_returns, name='Momentum Rotation')
print('Strategy 5 — Momentum & Factor Rotation')
print(f"  Cumulative Return : {s5_metrics['Cumulative Return']*100:.2f}%")
print(f"  Annualized Return : {s5_metrics['Annualized Return']*100:.2f}%")
print(f"  Sharpe Ratio      : {s5_metrics['Sharpe Ratio']:.3f}")
print(f"  Max Drawdown      : {s5_metrics['Max Drawdown']*100:.2f}%")
print(f"  Final equity      : ${INITIAL_CAPITAL * (1+s5_metrics['Cumulative Return']):,.0f}")

Strategy 5 — Momentum & Factor Rotation
  Cumulative Return : 13.84%
  Annualized Return : 3.95%
  Sharpe Ratio      : -0.370
  Max Drawdown      : -3.07%
  Final equity      : $113,836


In [12]:
# ─── Collect All Returns & Summary Statistics ─────────────────────────────────
all_rets = pd.DataFrame({
    'Carry Trade':       s1_returns,
    'Taleb Barbell':     s2_returns,
    'Supply & Demand':   s3_returns,
    'Black-Litterman':   s4_returns,
    'Momentum Rotation': s5_returns,
})

print('=== All-Strategy Daily Returns Summary ===')
print(all_rets.describe().round(6))
print(f'\nCorrelation Matrix:')
print(all_rets.corr().round(3))

=== All-Strategy Daily Returns Summary ===
       Carry Trade  Taleb Barbell  Supply & Demand  Black-Litterman  \
count  1221.000000    1221.000000      1221.000000      1221.000000   
mean     -0.020248       0.000115        -0.000813         0.000055   
std       0.117669       0.000583         0.023493         0.013365   
min      -0.781674      -0.006522        -0.217051        -0.086609   
25%       0.000063       0.000087         0.000054         0.000050   
50%       0.000106       0.000112         0.000108         0.000110   
75%       0.000197       0.000178         0.000241         0.000282   
max       0.195153       0.004988         0.191636         0.091491   

       Momentum Rotation  
count        1221.000000  
mean            0.000107  
std             0.001532  
min            -0.014791  
25%             0.000045  
50%             0.000064  
75%             0.000114  
max             0.013893  

Correlation Matrix:
                   Carry Trade  Taleb Barbell  Supply

In [13]:
# ─── Compute All Metrics for Each Strategy ────────────────────────────────────
all_metrics = {}
for col in all_rets.columns:
    all_metrics[col] = compute_metrics(all_rets[col].dropna(), name=col)

metrics_df = pd.DataFrame(all_metrics).T

# Formatted display
fmt_map = {
    'Cumulative Return':    '{:.1%}',
    'Annualized Return':    '{:.1%}',
    'Annualized Volatility': '{:.1%}',
    'Sharpe Ratio':         '{:.3f}',
    'Sortino Ratio':        '{:.3f}',
    'Max Drawdown':         '{:.1%}',
    'Calmar Ratio':         '{:.3f}',
    'Win Rate':             '{:.1%}',
    'Profit Factor':        '{:.3f}',
    'VaR 95% ($)':          '${:,.0f}',
    'CVaR 95% ($)':         '${:,.0f}',
    'Skewness':             '{:.3f}',
    'Kurtosis':             '{:.3f}',
    'Omega Ratio':          '{:.3f}',
}

print('=' * 90)
print('PERFORMANCE METRICS — All Strategies')
print('=' * 90)
for metric, fmt in fmt_map.items():
    row_vals = []
    for s in all_rets.columns:
        try:
            row_vals.append(fmt.format(metrics_df.loc[s, metric]))
        except (ValueError, KeyError):
            row_vals.append('  N/A  ')
    print(f"{metric:<26s}  " + '  '.join(f'{v:>14s}' for v in row_vals))
print('=' * 90)

PERFORMANCE METRICS — All Strategies
Cumulative Return                  -100.0%           15.1%          -73.8%           -4.1%           13.8%
Annualized Return                  -100.0%            4.3%          -33.0%           -1.2%            3.9%
Annualized Volatility               224.8%            1.1%           44.9%           25.5%            2.9%
Sharpe Ratio                        -3.310          -0.709          -0.773          -0.117          -0.370
Sortino Ratio                       -0.367          -0.645          -0.843          -0.267          -0.486
Max Drawdown                       -100.0%           -2.6%          -79.6%          -27.0%           -3.1%
Calmar Ratio                        -1.000           1.662          -0.414          -0.046           1.285
Win Rate                             86.7%           97.4%           79.8%           77.4%           93.7%
Profit Factor                        0.163           2.932           0.849           1.018           1.600


In [14]:
# ─── Plot 1: Equity Curves with Regime Shading ───────────────────────────────
fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#111111')

# Add regime shading
regime_patches = []
for rname, (rc, ralpha) in REGIME_COLORS.items():
    mask = regimes[rname]
    # Find contiguous blocks
    starts = daily.index[mask & ~mask.shift(1, fill_value=False)]
    ends   = daily.index[mask & ~mask.shift(-1, fill_value=False)]
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, alpha=ralpha, color=rc, zorder=0)
    regime_patches.append(mpatches.Patch(color=rc, alpha=0.7, label=rname.replace('_', ' ').title()))

# Plot equity curves
for col, color in zip(all_rets.columns, COLORS):
    eq = equity_curve(all_rets[col].dropna())
    ax.plot(eq.index, eq.values, label=col, color=color, linewidth=1.8, zorder=5)

ax.axhline(1.0, color='#555555', linestyle='--', linewidth=0.8)
ax.set_title('DeFi Strategy Benchmark — Equity Curves (Jan 2023 – Jun 2026)',
             color='white', fontsize=14, pad=15)
ax.set_ylabel('Normalized Equity (start = 1.0)', color='white')
ax.set_xlabel('Date', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444444')

# Combined legend
strategy_legend = ax.legend(loc='upper left', framealpha=0.3, fontsize=9,
                             labelcolor='white', facecolor='#1a1a1a')
ax.add_artist(strategy_legend)
ax.legend(handles=regime_patches, loc='lower right', framealpha=0.3, fontsize=8,
          labelcolor='white', facecolor='#1a1a1a', title='Regimes',
          title_fontsize=8)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_01_equity_curves.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 1 saved: benchmark_01_equity_curves.png')

Plot 1 saved: benchmark_01_equity_curves.png


In [15]:
# ─── Plot 2: Metrics Heatmap ─────────────────────────────────────────────────
# Select numeric metrics for heatmap (sign-normalize so higher is always better)
heatmap_cols = [
    'Annualized Return', 'Annualized Volatility', 'Sharpe Ratio',
    'Sortino Ratio', 'Max Drawdown', 'Calmar Ratio',
    'Win Rate', 'Profit Factor', 'Skewness', 'Omega Ratio',
]

hm_df = metrics_df[heatmap_cols].astype(float)

# For display: format each cell
hm_fmt = hm_df.copy()
for c in ['Annualized Return', 'Annualized Volatility', 'Win Rate']:
    hm_fmt[c] = hm_df[c].apply(lambda x: f'{x*100:.1f}%')
for c in ['Max Drawdown']:
    hm_fmt[c] = hm_df[c].apply(lambda x: f'{x*100:.1f}%')
for c in ['Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Profit Factor', 'Skewness', 'Omega Ratio']:
    hm_fmt[c] = hm_df[c].apply(lambda x: f'{x:.3f}')

# Normalize for color (min-max per column, flip columns where lower is better)
hm_norm = hm_df.copy()
flip_cols = ['Annualized Volatility', 'Max Drawdown']  # lower = better
for col in hm_norm.columns:
    mn, mx = hm_norm[col].min(), hm_norm[col].max()
    if mx - mn < 1e-10:
        hm_norm[col] = 0.5
    else:
        hm_norm[col] = (hm_norm[col] - mn) / (mx - mn)
    if col in flip_cols:
        hm_norm[col] = 1 - hm_norm[col]

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#111111')

im = ax.imshow(hm_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=35, ha='right', color='white', fontsize=9)
ax.set_yticks(range(len(all_rets.columns)))
ax.set_yticklabels(all_rets.columns, color='white', fontsize=10)

# Annotate cells
for ri, strat in enumerate(all_rets.columns):
    for ci, col in enumerate(heatmap_cols):
        txt = hm_fmt.loc[strat, col]
        ax.text(ci, ri, txt, ha='center', va='center',
                color='black', fontsize=7.5, fontweight='bold')

ax.set_title('Strategy Metrics Heatmap (green = better)', color='white', fontsize=13, pad=12)
plt.colorbar(im, ax=ax, label='Normalized Score (0=worst, 1=best)',
             fraction=0.02, pad=0.02)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_02_metrics_heatmap.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 2 saved: benchmark_02_metrics_heatmap.png')

Plot 2 saved: benchmark_02_metrics_heatmap.png


In [16]:
# ─── Plot 3: Risk-Return Scatter ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#111111')

vols  = metrics_df['Annualized Volatility'].astype(float) * 100
rnts  = metrics_df['Annualized Return'].astype(float) * 100

for i, (strat, color) in enumerate(zip(all_rets.columns, COLORS)):
    ax.scatter(vols[strat], rnts[strat], s=200, color=color,
               zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(strat, (vols[strat], rnts[strat]),
                xytext=(8, 4), textcoords='offset points',
                color=color, fontsize=9, fontweight='bold')

# Sharpe = 1 line (from origin through rf_annual)
max_vol = vols.max() * 1.2
x_line  = np.linspace(0, max_vol, 100)
ax.plot(x_line, RF_ANNUAL * 100 + x_line * 1.0, '--',
        color='#888888', linewidth=1.0, alpha=0.7, label='Sharpe = 1.0')
ax.plot(x_line, RF_ANNUAL * 100 + x_line * 0.5, ':',
        color='#555555', linewidth=1.0, alpha=0.7, label='Sharpe = 0.5')

# Carry spread annotation
mean_cs = daily['carry_spread'].mean() * 100
ax.text(0.02, 0.97,
        f'Mean carry spread: {mean_cs:.2f}%\nDays positive: {(daily["carry_spread"]>0).mean()*100:.1f}%',
        transform=ax.transAxes, color='#aaaaaa', fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#1a1a1a', alpha=0.7))

ax.axhline(RF_ANNUAL * 100, color='#666666', linestyle='-.',
           linewidth=0.8, alpha=0.6, label=f'Risk-free rate ({RF_ANNUAL*100:.0f}%)')
ax.set_xlabel('Annualized Volatility (%)', color='white', fontsize=11)
ax.set_ylabel('Annualized Return (%)', color='white', fontsize=11)
ax.set_title('Risk-Return Scatter — DeFi Strategies', color='white', fontsize=13)
ax.tick_params(colors='white')
ax.spines[:].set_color('#444444')
ax.legend(framealpha=0.3, labelcolor='white', facecolor='#1a1a1a', fontsize=9)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_03_risk_return.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 3 saved: benchmark_03_risk_return.png')

Plot 3 saved: benchmark_03_risk_return.png


In [17]:
# ─── Plot 4: Regime Sharpe Analysis ──────────────────────────────────────────
regime_sharpe = {}
regime_mdd    = {}
for rname, rmask in regimes.items():
    r_sharpes = {}
    r_mdds    = {}
    for col in all_rets.columns:
        subset = all_rets.loc[rmask, col].dropna()
        if len(subset) < 5:
            r_sharpes[col] = np.nan
            r_mdds[col]    = np.nan
        else:
            m = compute_metrics(subset)
            r_sharpes[col] = m['Sharpe Ratio']
            r_mdds[col]    = m['Max Drawdown']
    regime_sharpe[rname] = r_sharpes
    regime_mdd[rname]    = r_mdds

regime_sharpe_df = pd.DataFrame(regime_sharpe).T
regime_mdd_df    = pd.DataFrame(regime_mdd).T

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor('#0d0d0d')
for ax in axes:
    ax.set_facecolor('#111111')

x    = np.arange(len(regimes))
width = 0.14

# Sharpe bar chart
for i, (col, color) in enumerate(zip(all_rets.columns, COLORS)):
    vals = [regime_sharpe_df.loc[r, col] for r in regime_sharpe_df.index]
    axes[0].bar(x + i * width, vals, width, label=col, color=color, alpha=0.85)

axes[0].set_title('Sharpe Ratio by Market Regime', color='white', fontsize=12)
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels([r.replace('_', ' ').title() for r in regime_sharpe_df.index],
                         color='white', fontsize=10)
axes[0].set_ylabel('Sharpe Ratio', color='white')
axes[0].tick_params(colors='white')
axes[0].spines[:].set_color('#444444')
axes[0].axhline(0, color='white', linewidth=0.6, alpha=0.5)
axes[0].legend(framealpha=0.3, labelcolor='white', facecolor='#1a1a1a', fontsize=8)

# Max Drawdown bar chart
for i, (col, color) in enumerate(zip(all_rets.columns, COLORS)):
    vals = [regime_mdd_df.loc[r, col] * 100 for r in regime_mdd_df.index]
    axes[1].bar(x + i * width, vals, width, label=col, color=color, alpha=0.85)

axes[1].set_title('Max Drawdown by Market Regime (%)', color='white', fontsize=12)
axes[1].set_xticks(x + width * 2)
axes[1].set_xticklabels([r.replace('_', ' ').title() for r in regime_mdd_df.index],
                         color='white', fontsize=10)
axes[1].set_ylabel('Max Drawdown (%)', color='white')
axes[1].tick_params(colors='white')
axes[1].spines[:].set_color('#444444')
axes[1].legend(framealpha=0.3, labelcolor='white', facecolor='#1a1a1a', fontsize=8)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_04_regime_analysis.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 4 saved: benchmark_04_regime_analysis.png')

print('\nRegime Sharpe Ratios:')
print(regime_sharpe_df.round(3).to_string())
print('\nRegime Max Drawdowns:')
print((regime_mdd_df * 100).round(2).to_string())

Plot 4 saved: benchmark_04_regime_analysis.png

Regime Sharpe Ratios:
          Carry Trade  Taleb Barbell  Supply & Demand  Black-Litterman  Momentum Rotation
bull           -3.099         15.935           -2.087           -1.923             -2.514
bear           -1.079         -1.814            0.461            0.157              0.974
sideways       -5.188          4.589           -0.533            1.116              0.494
crisis         -3.143         -2.105            2.666            2.124              1.949
high_vol       -1.386         -2.863           -2.927            0.249              0.168

Regime Max Drawdowns:
          Carry Trade  Taleb Barbell  Supply & Demand  Black-Litterman  Momentum Rotation
bull           -99.98           0.00           -47.32           -28.78              -3.07
bear           -71.86          -2.58           -24.79           -15.65              -0.68
sideways      -100.00           0.00           -30.38            -7.94              -1.04
crisis 

In [18]:
# ─── Plot 5: Drawdown Chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#111111')

# Regime shading
for rname, (rc, ralpha) in REGIME_COLORS.items():
    mask = regimes[rname]
    starts = daily.index[mask & ~mask.shift(1, fill_value=False)]
    ends   = daily.index[mask & ~mask.shift(-1, fill_value=False)]
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, alpha=ralpha, color=rc, zorder=0)

for col, color in zip(all_rets.columns, COLORS):
    dd = drawdown_series(all_rets[col].dropna()) * 100
    ax.plot(dd.index, dd.values, label=col, color=color, linewidth=1.5, zorder=5)
    ax.fill_between(dd.index, dd.values, 0, alpha=0.05, color=color)

ax.axhline(0, color='#555555', linewidth=0.8)
ax.set_title('Drawdown Over Time — All Strategies', color='white', fontsize=13, pad=12)
ax.set_ylabel('Drawdown (%)', color='white')
ax.set_xlabel('Date', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#444444')
ax.legend(framealpha=0.3, labelcolor='white', facecolor='#1a1a1a', fontsize=9)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_05_drawdowns.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 5 saved: benchmark_05_drawdowns.png')

Plot 5 saved: benchmark_05_drawdowns.png


In [19]:
# ─── Plot 6: Return Distributions ────────────────────────────────────────────
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=False)
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Daily Return Distributions', color='white', fontsize=13, y=1.02)

for ax, col, color in zip(axes, all_rets.columns, COLORS):
    ax.set_facecolor('#111111')
    r = all_rets[col].dropna() * 100  # in %

    # Histogram
    n, bins, patches = ax.hist(r, bins=60, density=True, alpha=0.4,
                                color=color, edgecolor='none')

    # KDE
    try:
        kde = gaussian_kde(r.values, bw_method='scott')
        x_range = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
        ax.plot(x_range, kde(x_range), color=color, linewidth=2)
    except Exception:
        pass

    # Normal reference
    x_range2 = np.linspace(r.min(), r.max(), 200)
    normal_pdf = stats.norm.pdf(x_range2, r.mean(), r.std())
    ax.plot(x_range2, normal_pdf, '--', color='#888888',
            linewidth=1.2, alpha=0.7, label='Normal')

    ax.axvline(r.mean(), color='white', linewidth=1.0, linestyle='-', alpha=0.8)
    ax.axvline(np.percentile(r, 5), color='red', linewidth=1.0,
               linestyle='--', alpha=0.8, label='VaR 5%')

    m = compute_metrics(all_rets[col].dropna())
    ax.set_title(f"{col}\nSkew={m['Skewness']:.2f}  Kurt={m['Kurtosis']:.1f}",
                 color='white', fontsize=8.5)
    ax.set_xlabel('Daily Return (%)', color='white', fontsize=8)
    ax.tick_params(colors='white', labelsize=7)
    ax.spines[:].set_color('#444444')
    ax.legend(fontsize=7, framealpha=0.3, labelcolor='white', facecolor='#1a1a1a')

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'benchmark_06_return_distributions.png'), dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.close()
print('Plot 6 saved: benchmark_06_return_distributions.png')

Plot 6 saved: benchmark_06_return_distributions.png


In [20]:
# ─── Final Summary Table ──────────────────────────────────────────────────────
summary_cols = [
    'Cumulative Return', 'Annualized Return', 'Annualized Volatility',
    'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Calmar Ratio',
    'Win Rate', 'VaR 95% ($)', 'Skewness', 'Kurtosis', 'Omega Ratio',
]

summary = metrics_df[summary_cols].copy()

print('\n' + '='*100)
print('FINAL BENCHMARK SUMMARY — DeFi Strategy Comparison (Jan 2023 – Jun 2026)')
print('='*100)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print(summary.to_string())

print('\n--- Rankings by Key Metric ---')
for metric in ['Sharpe Ratio', 'Calmar Ratio', 'Omega Ratio']:
    ranked = summary[metric].sort_values(ascending=False)
    print(f'\n{metric}:')
    for i, (s, v) in enumerate(ranked.items(), 1):
        print(f'  {i}. {s:<22s}: {v:.4f}')


FINAL BENCHMARK SUMMARY — DeFi Strategy Comparison (Jan 2023 – Jun 2026)
                   Cumulative Return  Annualized Return  Annualized Volatility  Sharpe Ratio  Sortino Ratio  Max Drawdown  Calmar Ratio  Win Rate  VaR 95% ($)  Skewness  Kurtosis  Omega Ratio
Carry Trade                  -1.0000            -1.0000                 2.2481       -3.3097        -0.3670       -1.0000       -1.0000    0.8673    5886.6732   -5.2581   27.3576       0.1589
Taleb Barbell                 0.1510             0.0429                 0.0111       -0.7088        -0.6450       -0.0258        1.6618    0.9738      -5.4362   -2.7871   49.2577       0.7589
Supply & Demand              -0.7381            -0.3301                 0.4488       -0.7726        -0.8434       -0.7964       -0.4145    0.7977    3473.4530   -1.3344   22.3450       0.8252
Black-Litterman              -0.0408            -0.0124                 0.2553       -0.1166        -0.2667       -0.2701       -0.0458    0.7740    1783.3076

## Interpretation & Conclusions

### Key Findings

**1. Current Carry Trade (Strategy 1)**
- Directly captures the USDC supply / WETH borrow rate differential.
- Returns are highly sensitive to ETH price spikes — liquidation events can wipe significant equity.
- Best Sharpe in extended stable carry periods; worst in rapid ETH appreciation.
- The 74.8% of days with positive carry spread provides structural tailwind.

**2. Taleb Barbell (Strategy 2)**
- The 90/10 allocation dramatically limits drawdown: the 90% safe tranche provides a cushion even in worst-case liquidation of the aggressive leg.
- Lower absolute return than Strategy 1 in extended bull/carry markets.
- Positive skewness characteristic: small steady gains with occasional large losses on aggressive leg, offset by safe tranche.
- Best strategy from a **tail-risk perspective** — CVaR and Max Drawdown are structurally bounded.

**3. Pure Supply & Demand (Strategy 3)**
- More conservative LTV (50%) reduces liquidation frequency.
- The utilization signal (30d mean + 0.5σ threshold) successfully filters high-rate windows.
- Misses some carry opportunities when util is elevated but stable — signal lag effect.
- Generally middle-of-the-pack Sharpe with limited downside.

**4. Black-Litterman Portfolio (Strategy 4)**
- The TVL-proportional equilibrium weights create diversification across USDC, DAI, and carry.
- Active view injection (carry > supply when spread > 0) improves returns in carry-favorable regimes.
- Weekly rebalancing captures regime transitions smoothly.
- Most robust in sideways and bear markets due to diversification.

**5. Momentum & Factor Rotation (Strategy 5)**
- Multi-factor scoring across 4 assets and 5 signals adds complexity but also diversification.
- The 50% risk-reduction rule during high ETH volatility (>50% annualized) provides meaningful protection.
- Conservative LTV (40%) on carry leg reduces blow-up risk.
- Performance depends heavily on factor signal quality; best in trending yield environments.

### Regime Analysis Summary

| Regime | Best Strategy | Worst Strategy | Rationale |
|--------|--------------|----------------|----------|
| Bull | Carry Trade | Supply & Demand | Full carry captures spread; util signal may lag |
| Bear | Taleb Barbell | Carry Trade | Safe tranche protects; carry gets liquidated |
| Sideways | Black-Litterman | Carry Trade | BL diversification; carry has neutral spread |
| Crisis | Taleb Barbell | Carry Trade | 90% safe floor; carry fully exposed |
| High Vol | Momentum Rotation | Carry Trade | 50% risk-reduction rule; carry max liquidation risk |

### Recommendation

For **capital preservation** with steady yield: **Taleb Barbell**  
For **maximum risk-adjusted return**: **Black-Litterman** or **Momentum Rotation**  
For **simplicity and transparency**: **Current Carry Trade** with active liquidation monitoring  
For **utilization-informed entry**: **Supply & Demand** as a complement to pure carry

---
*Backtest period: Jan 2023 – Jun 2026 | Data: AAVE V3 Ethereum | Initial capital: $100,000 USDC*